# Load PRD model from MLflow and predict

A clean, standalone notebook. Does not retrain. Loads `models:/chronos_1h_prd@prd` and makes test predictions.

**Prerequisites:** `experiment_tracking.ipynb` must have run and registered the PRD model.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, str(pathlib.Path('.').resolve()))

import mlflow
import pandas as pd
from chronos_ts.tracking import configure_mlflow

configure_mlflow()

In [ ]:
# Load the PRD model
REGISTERED_MODEL = 'chronos_1h_prd'
model = mlflow.pyfunc.load_model(f'models:/{REGISTERED_MODEL}@prd')
print(f'Loaded: models:/{REGISTERED_MODEL}@prd')
print(f'Flavors: {list(model.metadata.flavors.keys())}')

In [ ]:
# Reconstruct test features (same split as training)
from chronos_ts.labels import LabelConfig, LabelMaker
from chronos_ts.splits import TimeRangeSplitConfig, time_fraction_split
import json

METRICS_PATH = 'outputs/models/clf/catboost_vol_regime/catboost_vol_regime_metrics.json'
metrics = json.load(open(METRICS_PATH))
feature_cols = metrics['feature_cols']

df = pd.read_csv('outputs/datasets/btcusdt_clf_core.csv', parse_dates=['ts'])
df = df.sort_values('ts').reset_index(drop=True)

split_cfg = TimeRangeSplitConfig(train_frac=0.70, val_frac=0.15, test_frac=0.15)
splits = time_fraction_split(df, split_cfg, ts_col='ts')

lm = LabelMaker(LabelConfig(target_family='vol_regime'))
lm.fit(splits['train'])

y_test = lm.transform(splits['test'])
mask = y_test.notna()
X_test = splits['test'].loc[mask, feature_cols].reset_index(drop=True)

print(f'Test set: {len(X_test)} rows, {len(feature_cols)} features')

In [ ]:
# Predict on last 10 rows
sample = X_test.tail(10)
preds = model.predict(sample)

class_names = lm.class_names()
y_true_sample = y_test[mask].values[-10:].astype(int)

result_df = pd.DataFrame({
    'ts': splits['test'].loc[mask, 'ts'].values[-10:],
    'y_true': [class_names[i] for i in y_true_sample],
    'y_pred': [class_names[int(p)] for p in preds],
    'correct': y_true_sample == preds.astype(int),
})
print(result_df.to_string(index=False))

In [ ]:
# Smoke check: predictions on full test set must match saved CSV
import numpy as np

saved_preds = pd.read_csv('outputs/models/clf/catboost_vol_regime/catboost_vol_regime_test_predictions.csv')
full_preds = model.predict(X_test)

match_rate = (full_preds.astype(int) == saved_preds['y_pred'].values).mean()
print(f'Match rate vs saved predictions: {match_rate:.4f}')
assert match_rate >= 0.99, f'Predictions diverged! match_rate={match_rate}'
print('✅ Reproducibility check passed — PRD model predictions are stable')